In [ ]:
import pandas as pd

train = pd.read_json("train.json")
test  = pd.read_json("test.json")

print(train.head())
print(train.columns)

In [ ]:
train.info()
train.isna().sum()
train["type"].value_counts()

In [ ]:
train["text_len"] = train["answer"].apply(lambda x: len(str(x).split()))
train["text_len"].hist(bins=30)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = " ".join(train["answer"])
wc = WordCloud(width=800, height=400, background_color="white").generate(text)
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")

In [ ]:
import re
from sklearn.model_selection import train_test_split

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s]", "", t)
    return t

train["clean"] = train["answer"].apply(clean_text)
X = train["clean"]
y = train["type"]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_vec = tfidf.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_valid, y_train, y_valid = train_test_split(X_vec, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

pred = model.predict(X_valid)
print(classification_report(y_valid, pred))
# f1 score

In [ ]:
test["clean"] = test["answer"].apply(clean_text)
X_test = tfidf.transform(test["clean"])
test["type"] = model.predict(X_test)